In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from utils import preprocess_data, get_features_and_target
from classification_utils import (
    train_and_compare_classification,
    print_results_table,
    print_confusion_matrix
)

In [2]:
#Подготовка данных
df = preprocess_data('data.xlsx')

# Получаем признаки и бинарную целевую переменную
X, y = get_features_and_target(df, 'CC50, mM',
                                task_type='classification',
                                threshold=None)

# Считаем медиану для информации
median_val = df['CC50, mM'].median()
print(f"\nЦелевая переменная: CC50 > медианы (медиана = {median_val:.3f})")
print(f"Размер X: {X.shape}, размер y: {y.shape}")
print(f"\nБаланс классов:")
print(f"  Класс 0 (CC50 <= медианы): {(y == 0).sum()} ({(y == 0).mean()*100:.1f}%)")
print(f"  Класс 1 (CC50 > медианы):  {(y == 1).sum()} ({(y == 1).mean()*100:.1f}%)")

Загружено: 1001 объектов, 213 столбцов
Удалено константных признаков: 33
Пропуски заполнены медианой каждого столбца
Удалено признаков с корреляцией > 0.95: 33
Итого: 1001 объектов, 147 столбцов (144 признаков)

Целевая переменная: CC50 > медианы (медиана = 411.039)
Размер X: (1001, 144), размер y: (1001,)

Баланс классов:
  Класс 0 (CC50 <= медианы): 502 (50.1%)
  Класс 1 (CC50 > медианы):  499 (49.9%)


In [3]:
#обучение моделей
results_df, best_models, splits = train_and_compare_classification(
    X, y,
    target_name='CC50 > медианы',
    cv=5,
    test_size=0.3,
    random_state=42
)

X_train, X_test, y_train, y_test = splits

In [4]:
#Вывод таблицы
print_results_table(results_df, target_name='CC50 > медианы')

from pathlib import Path; Path('results').mkdir(parents=True, exist_ok=True)
results_df.to_csv('results/results_classification_CC50_median.csv', index=False)


             Model    CV F1  Test Accuracy  Test Precision  Test Recall  Test F1  Test ROC_AUC
      RandomForest 0.749434       0.740864        0.704545     0.826667 0.760736      0.830265
  GradientBoosting 0.756042       0.737542        0.707602     0.806667 0.753894      0.839492
      DecisionTree 0.738193       0.740864        0.722222     0.780000 0.750000      0.753841
LogisticRegression 0.752071       0.717608        0.687861     0.793333 0.736842      0.796402
               KNN 0.775457       0.704319        0.670391     0.800000 0.729483      0.788168
               SVC 0.790226       0.710963        0.688623     0.766667 0.725552      0.806380

Лучшая модель: RandomForest (Test F1 = 0.7607)


In [5]:
#Визуализация лучшей модели
# Сравнение моделей по F1-score
plt.figure(figsize=(12, 6))
sorted_df = results_df.sort_values('Test F1', ascending=True)
plt.barh(sorted_df['Model'], sorted_df['Test F1'], color='steelblue', edgecolor='black')
plt.xlabel('F1-score на тестовой выборке')
plt.title('Сравнение моделей классификации по F1-score (CC50 > медианы)')
plt.axvline(x=0.5, color='red', linestyle='--', linewidth=1, label='Случайное угадывание')
plt.legend()
plt.tight_layout()
plt.savefig('results/comparison_classification_CC50_median.png',
            dpi=100, bbox_inches='tight')
plt.close()

# Подробный анализ лучшей модели
best_model_name = results_df.iloc[0]['Model']
best_model = best_models[best_model_name]

y_pred_test = best_model.predict(X_test)

# Матрица ошибок
print_confusion_matrix(y_test, y_pred_test, model_name=best_model_name)

# Визуализация матрицы ошибок
cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Класс 0', 'Класс 1'],
            yticklabels=['Класс 0', 'Класс 1'])
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.title(f'Матрица ошибок: {best_model_name}\n(CC50 > медианы)')
plt.tight_layout()
plt.savefig('results/confusion_matrix_CC50_median.png', dpi=100, bbox_inches='tight')
plt.close()


TN (правильно отрицательные): 99
FP (ложные срабатывания):     52
FN (пропущенные положительные): 26
TP (правильно положительные): 124
              precision    recall  f1-score   support

           0       0.79      0.66      0.72       151
           1       0.70      0.83      0.76       150

    accuracy                           0.74       301
   macro avg       0.75      0.74      0.74       301
weighted avg       0.75      0.74      0.74       301



In [6]:
#Важность признаков
final_model = best_model.named_steps['model']
if hasattr(final_model, 'feature_importances_'):
    print(f"\nТоп-15 важных признаков для {best_model_name}")
    importances = pd.Series(final_model.feature_importances_, index=X.columns)
    top_features = importances.sort_values(ascending=False).head(15)
    for feat, imp in top_features.items():
        print(f"  {feat:30s}: {imp:.4f}")

    plt.figure(figsize=(10, 6))
    top_features.sort_values().plot(kind='barh', color='coral', edgecolor='black')
    plt.xlabel('Важность признака')
    plt.title(f'Топ-15 признаков для классификации CC50 > медианы\n({best_model_name})')
    plt.tight_layout()
    plt.savefig('results/feature_importance_CC50_median.png',
                dpi=100, bbox_inches='tight')
    plt.close()



Топ-15 важных признаков для RandomForest
  BCUT2D_MWLOW                  : 0.0311
  NHOHCount                     : 0.0253
  PEOE_VSA7                     : 0.0236
  NumSaturatedHeterocycles      : 0.0217
  VSA_EState4                   : 0.0207
  MinPartialCharge              : 0.0204
  VSA_EState7                   : 0.0181
  MolLogP                       : 0.0178
  BCUT2D_LOGPHI                 : 0.0172
  Kappa3                        : 0.0165
  BCUT2D_CHGLO                  : 0.0161
  BalabanJ                      : 0.0160
  MaxPartialCharge              : 0.0151
  BCUT2D_MRLOW                  : 0.0151
  SPS                           : 0.0150
